In [1]:
!pip uninstall -y pgmpy
!pip install pgmpy==0.1.23

Found existing installation: pgmpy 0.1.23
Uninstalling pgmpy-0.1.23:
  Successfully uninstalled pgmpy-0.1.23
  Using cached pgmpy-0.1.23-py3-none-any.whl.metadata (6.3 kB)
Using cached pgmpy-0.1.23-py3-none-any.whl (1.9 MB)



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pgmpy
import sys

print("Python version:", sys.version)
print("pgmpy version:", pgmpy.__version__)

Python version: 3.9.11 (tags/v3.9.11:2de452f, Mar 16 2022, 14:33:45) [MSC v.1929 64 bit (AMD64)]
pgmpy version: 0.1.23


In [3]:
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

print("Bayesian Network libraries imported successfully!")

c:\Users\Anushree\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Bayesian Network libraries imported successfully!


In [5]:
import pandas as pd
import numpy as np
import json
import warnings
import sys

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler, OrdinalEncoder

from pgmpy.models import BayesianNetwork
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination

warnings.filterwarnings("ignore")

print("Python Version:", sys.version)
print("All libraries imported successfully!")

Python Version: 3.9.11 (tags/v3.9.11:2de452f, Mar 16 2022, 14:33:45) [MSC v.1929 64 bit (AMD64)]
All libraries imported successfully!


In [7]:
INPUT_FILE = "..\\data\\authorization\\preprocessed_authorization_data.csv"

df = pd.read_csv(INPUT_FILE)

# Keep an untouched copy
df_original = df.copy()

print("Dataset loaded successfully")
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

Dataset loaded successfully
Shape: (15000, 17)

Columns:
['authorization_id', 'patient_id', 'provider_id', 'request_date', 'authorization_type', 'service_code', 'service_description', 'requested_quantity', 'charged_amount', 'approval_status', 'approval_date', 'valid_from_date', 'valid_to_date', 'reference_number', 'payer_id', 'diagnosis_code', 'notes']


,authorization_id,patient_id,provider_id,request_date,authorization_type,service_code,service_description,requested_quantity,charged_amount,approval_status,approval_date,valid_from_date,valid_to_date,reference_number,payer_id,diagnosis_code,notes
0,AUTH00001-2026A,PAT58368,PRV4993,2025-03-31,medication,NDC43598-0545-01,Albuterol inhaler,2,187.35,approved,2025-04-04,2025-04-01,2025-05-01,REF-PA-350001,PAY7686,J45.909,Request submitted for review
1,AUTH00002-2026A,PAT25525,PRV8245,2025-12-05,DME,HCPCS E0165,Commode chair,1,338.48,approved,2025-12-11,2025-12-07,2026-06-05,REF-PA-350002,PAY6942,R26.89,Request submitted for review
2,AUTH00003-2026A,PAT35112,PRV3165,2026-02-04,procedure,CPT99214,"Office visit, established patient",1,362.78,approved,2026-02-06,2026-02-06,2026-08-05,REF-PA-350003,PAY2627,E78.5,Standard authorization request
3,AUTH00004-2026A,PAT42177,PRV4073,2026-04-14,DME,HCPCS E0607,CPAP device,1,2007.68,denied,2026-04-16,2026-04-16,2026-06-15,REF-PA-350004,PAY7367,G47.33,Medical necessity criteria not met
4,AUTH00005-2026A,PAT71433,PRV3634,2025-01-27,procedure,CPT29881,Knee arthroscopy with meniscectomy,1,5304.87,approved,2025-02-03,2025-01-27,2025-03-28,REF-PA-350005,PAY8432,M23.21,Clinical review in progress


In [8]:
print("DATA TYPES")
print(df.dtypes)

print("\n" + "=" * 60)
print("MISSING VALUES")
print("=" * 60)

missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

display(
    missing_summary.sort_values(
        "missing_count",
        ascending=False
    )
)

print("\nDUPLICATE ROWS:", df.duplicated().sum())

DATA TYPES
authorization_id        object
patient_id              object
provider_id             object
request_date            object
authorization_type      object
service_code            object
service_description     object
requested_quantity       int64
charged_amount         float64
approval_status         object
approval_date           object
valid_from_date         object
valid_to_date           object
reference_number        object
payer_id                object
diagnosis_code          object
notes                   object
dtype: object

MISSING VALUES


,missing_count,missing_percentage
approval_date,2201,14.67
authorization_id,0,0.00
approval_status,0,0.00
diagnosis_code,0,0.00
payer_id,0,0.00
reference_number,0,0.00
valid_to_date,0,0.00
valid_from_date,0,0.00
charged_amount,0,0.00
patient_id,0,0.00



DUPLICATE ROWS: 0


In [9]:
date_cols = [
    "request_date",
    "approval_date",
    "valid_from_date",
    "valid_to_date"
]

dates = {}

for col in date_cols:
    dates[col] = pd.to_datetime(
        df[col],
        errors="coerce"
    )

print("Date conversion completed.")

Date conversion completed.


In [10]:
requested_quantity_num = pd.to_numeric(
    df["requested_quantity"],
    errors="coerce"
)

charged_amount_num = pd.to_numeric(
    df["charged_amount"],
    errors="coerce"
)

print("Numeric conversion completed.")

Numeric conversion completed.


In [11]:
features = pd.DataFrame(index=df.index)

# ==========================================
# 1. MISSING VALUE FEATURES
# ==========================================

critical_cols = [
    "patient_id",
    "provider_id",
    "request_date",
    "authorization_type",
    "service_code",
    "requested_quantity",
    "charged_amount",
    "approval_status"
]

features["missing_critical_count"] = (
    df[critical_cols]
    .isnull()
    .sum(axis=1)
)


# ==========================================
# 2. INVALID DATE COUNT
# ==========================================

invalid_date_count = pd.Series(
    0,
    index=df.index
)

for col in date_cols:

    invalid_date = (
        df[col].notna()
        &
        dates[col].isna()
    )

    invalid_date_count += invalid_date.astype(int)

features["invalid_date_count"] = invalid_date_count


# ==========================================
# 3. DATE-DIFFERENCE FEATURES
# ==========================================

features["approval_delay_days"] = (
    dates["approval_date"]
    -
    dates["request_date"]
).dt.days


features["validity_duration_days"] = (
    dates["valid_to_date"]
    -
    dates["valid_from_date"]
).dt.days


features["request_to_valid_days"] = (
    dates["valid_from_date"]
    -
    dates["request_date"]
).dt.days


# ==========================================
# 4. NUMERIC FEATURES
# ==========================================

features["requested_quantity"] = (
    requested_quantity_num
)

features["charged_amount"] = (
    charged_amount_num
)


print("Feature engineering completed.")
print("Feature shape:", features.shape)

display(features.head())

Feature engineering completed.
Feature shape: (15000, 7)


,missing_critical_count,invalid_date_count,approval_delay_days,validity_duration_days,request_to_valid_days,requested_quantity,charged_amount
0,0,0,4.0,30,1,2,187.35
1,0,0,6.0,180,2,1,338.48
2,0,0,2.0,180,2,1,362.78
3,0,0,2.0,60,2,1,2007.68
4,0,0,7.0,60,0,1,5304.87


In [12]:
for col in df.columns:
    
    features[f"{col}_missing_flag"] = (
        df[col].isnull()
    ).astype(int)

print("Missing value flags created.")
print("Total features:", features.shape[1])

Missing value flags created.
Total features: 24


In [13]:
conditions = pd.DataFrame(index=df.index)


# ==========================================
# MISSING DATA
# ==========================================

conditions["missing_data"] = np.where(
    features["missing_critical_count"] > 0,
    "yes",
    "no"
)


# ==========================================
# INVALID DATE FORMAT
# ==========================================

conditions["invalid_date"] = np.where(
    features["invalid_date_count"] > 0,
    "yes",
    "no"
)


# ==========================================
# FUTURE REQUEST DATE
# ==========================================

today = pd.Timestamp.today().normalize()

conditions["future_request"] = np.where(
    dates["request_date"] > today,
    "yes",
    "no"
)


# ==========================================
# APPROVAL BEFORE REQUEST
# ==========================================

conditions["approval_before_request"] = np.where(
    (
        dates["approval_date"].notna()
        &
        dates["request_date"].notna()
        &
        (
            dates["approval_date"]
            <
            dates["request_date"]
        )
    ),
    "yes",
    "no"
)


# ==========================================
# INVALID VALIDITY RANGE
# ==========================================

conditions["invalid_validity_range"] = np.where(
    (
        dates["valid_from_date"].notna()
        &
        dates["valid_to_date"].notna()
        &
        (
            dates["valid_to_date"]
            <
            dates["valid_from_date"]
        )
    ),
    "yes",
    "no"
)


# ==========================================
# NEGATIVE QUANTITY
# ==========================================

conditions["negative_quantity"] = np.where(
    requested_quantity_num < 0,
    "yes",
    "no"
)


# ==========================================
# NEGATIVE AMOUNT
# ==========================================

conditions["negative_amount"] = np.where(
    charged_amount_num < 0,
    "yes",
    "no"
)


print("Basic rule conditions created.")

display(conditions.head())

Basic rule conditions created.


,missing_data,invalid_date,future_request,approval_before_request,invalid_validity_range,negative_quantity,negative_amount
0,no,no,no,no,no,no,no
1,no,no,no,no,no,no,no
2,no,no,no,no,no,no,no
3,no,no,no,no,no,no,no
4,no,no,no,no,no,no,no


In [14]:
quantity_values = requested_quantity_num.dropna()

q1_quantity = quantity_values.quantile(0.25)
q3_quantity = quantity_values.quantile(0.75)

iqr_quantity = (
    q3_quantity - q1_quantity
)

quantity_lower = (
    q1_quantity - 1.5 * iqr_quantity
)

quantity_upper = (
    q3_quantity + 1.5 * iqr_quantity
)


conditions["unusual_quantity"] = np.where(
    (
        requested_quantity_num < quantity_lower
    )
    |
    (
        requested_quantity_num > quantity_upper
    ),
    "yes",
    "no"
)


print("Quantity lower limit:", quantity_lower)
print("Quantity upper limit:", quantity_upper)

Quantity lower limit: -2.0
Quantity upper limit: 6.0


In [15]:
amount_values = charged_amount_num.dropna()

q1_amount = amount_values.quantile(0.25)
q3_amount = amount_values.quantile(0.75)

iqr_amount = (
    q3_amount - q1_amount
)

amount_lower = (
    q1_amount - 1.5 * iqr_amount
)

amount_upper = (
    q3_amount + 1.5 * iqr_amount
)


conditions["unusual_amount"] = np.where(
    (
        charged_amount_num < amount_lower
    )
    |
    (
        charged_amount_num > amount_upper
    ),
    "yes",
    "no"
)


print("Amount lower limit:", amount_lower)
print("Amount upper limit:", amount_upper)

Amount lower limit: -3175.912499999999
Amount upper limit: 5621.487499999999


In [16]:
conditions["duplicate_record"] = np.where(
    df.duplicated(keep=False),
    "yes",
    "no"
)

print(
    conditions["duplicate_record"]
    .value_counts()
)

duplicate_record
no    15000
Name: count, dtype: int64


In [17]:
root_cause_cols = [
    "missing_data",
    "invalid_date",
    "future_request",
    "approval_before_request",
    "invalid_validity_range",
    "negative_quantity",
    "negative_amount",
    "unusual_quantity",
    "unusual_amount",
    "duplicate_record"
]


conditions["rule_anomaly_count"] = (
    conditions[root_cause_cols]
    .eq("yes")
    .sum(axis=1)
)


conditions["rule_anomaly_flag"] = np.where(
    conditions["rule_anomaly_count"] > 0,
    1,
    0
)


print("RULE-BASED ANOMALY SUMMARY")
print("=" * 60)

for col in root_cause_cols:
    print(
        f"{col}:",
        (conditions[col] == "yes").sum()
    )

print("\nTotal Rule-Based Anomalies:")
print(conditions["rule_anomaly_flag"].sum())

RULE-BASED ANOMALY SUMMARY
missing_data: 0
invalid_date: 0
future_request: 0
approval_before_request: 0
invalid_validity_range: 0
negative_quantity: 0
negative_amount: 0
unusual_quantity: 3531
unusual_amount: 765
duplicate_record: 0

Total Rule-Based Anomalies:
4296


In [18]:
ml_df = features.copy()

print("ML dataset shape:", ml_df.shape)

ML dataset shape: (15000, 24)


In [19]:
categorical_features = [
    "authorization_type",
    "approval_status",
    "service_description"
]

for col in categorical_features:

    ml_df[col] = (
        df[col]
        .fillna("__MISSING__")
        .astype(str)
        .str.lower()
        .str.strip()
    )

print("Categorical features added.")

Categorical features added.


In [20]:
ml_df["service_code_prefix"] = (
    df["service_code"]
    .fillna("__MISSING__")
    .astype(str)
    .str.extract(
        r"^([A-Za-z]+)",
        expand=False
    )
    .fillna("OTHER")
)

print(
    ml_df["service_code_prefix"]
    .value_counts()
)

service_code_prefix
CPT      7950
NDC      4258
HCPCS    2792
Name: count, dtype: int64


In [21]:
numeric_cols = ml_df.select_dtypes(
    include=[np.number]
).columns.tolist()


for col in numeric_cols:

    if ml_df[col].isna().any():

        median_value = ml_df[col].median()

        if pd.isna(median_value):
            median_value = 0

        ml_df[col] = ml_df[col].fillna(
            median_value
        )


print("Numeric missing values handled in ML copy.")

Numeric missing values handled in ML copy.


In [22]:
categorical_cols = ml_df.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()


encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)


if len(categorical_cols) > 0:

    ml_df[categorical_cols] = (
        encoder.fit_transform(
            ml_df[categorical_cols]
        )
    )


print("Categorical encoding completed.")
print("Encoded columns:", categorical_cols)

Categorical encoding completed.
Encoded columns: ['authorization_type', 'approval_status', 'service_description', 'service_code_prefix']


In [23]:
ml_df = ml_df.replace(
    [np.inf, -np.inf],
    np.nan
)


for col in ml_df.columns:

    if ml_df[col].isna().any():

        median_value = ml_df[col].median()

        if pd.isna(median_value):
            median_value = 0

        ml_df[col] = ml_df[col].fillna(
            median_value
        )


constant_cols = [
    col
    for col in ml_df.columns
    if ml_df[col].nunique() <= 1
]


ml_df = ml_df.drop(
    columns=constant_cols
)


print("Final ML dataset shape:", ml_df.shape)
print(
    "Remaining missing values:",
    ml_df.isnull().sum().sum()
)
print("Removed constant columns:", constant_cols)

Final ML dataset shape: (15000, 10)
Remaining missing values: 0
Removed constant columns: ['missing_critical_count', 'invalid_date_count', 'authorization_id_missing_flag', 'patient_id_missing_flag', 'provider_id_missing_flag', 'request_date_missing_flag', 'authorization_type_missing_flag', 'service_code_missing_flag', 'service_description_missing_flag', 'requested_quantity_missing_flag', 'charged_amount_missing_flag', 'approval_status_missing_flag', 'valid_from_date_missing_flag', 'valid_to_date_missing_flag', 'reference_number_missing_flag', 'payer_id_missing_flag', 'diagnosis_code_missing_flag', 'notes_missing_flag']


In [24]:
scaler = RobustScaler()

X_scaled = scaler.fit_transform(
    ml_df
)

print("Scaled data shape:", X_scaled.shape)

Scaled data shape: (15000, 10)


In [25]:
iso_model = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=42,
    n_jobs=-1
)


iso_predictions = iso_model.fit_predict(
    X_scaled
)


# Isolation Forest output:
# -1 = anomaly
#  1 = normal

iso_flag = np.where(
    iso_predictions == -1,
    1,
    0
)


iso_score = -iso_model.decision_function(
    X_scaled
)


print("ISOLATION FOREST RESULTS")
print("=" * 60)

print("Total records:", len(df))
print("ML anomalies:", iso_flag.sum())

print(
    "ML anomaly percentage:",
    round(iso_flag.mean() * 100, 2),
    "%"
)

ISOLATION FOREST RESULTS
Total records: 15000
ML anomalies: 750
ML anomaly percentage: 5.0 %


In [26]:
conditions["ml_anomaly_flag"] = iso_flag

conditions["ml_anomaly_score"] = iso_score

display(
    conditions[
        [
            "ml_anomaly_flag",
            "ml_anomaly_score"
        ]
    ].head()
)

,ml_anomaly_flag,ml_anomaly_score
0,0,-0.096876
1,0,-0.082698
2,0,-0.107327
3,0,-0.087366
4,0,-0.086950


In [27]:
conditions["final_anomaly_flag"] = np.where(
    (
        conditions["rule_anomaly_flag"] == 1
    )
    |
    (
        conditions["ml_anomaly_flag"] == 1
    ),
    1,
    0
)


print("FINAL ANOMALY SUMMARY")
print("=" * 60)

print(
    "Rule-Based Anomalies:",
    conditions["rule_anomaly_flag"].sum()
)

print(
    "ML Anomalies:",
    conditions["ml_anomaly_flag"].sum()
)

print(
    "Final Unique Anomalies:",
    conditions["final_anomaly_flag"].sum()
)

FINAL ANOMALY SUMMARY
Rule-Based Anomalies: 4296
ML Anomalies: 750
Final Unique Anomalies: 4497


In [28]:
bayesian_data = conditions[
    root_cause_cols
].copy()


bayesian_data["anomaly"] = np.where(
    conditions["ml_anomaly_flag"] == 1,
    "yes",
    "no"
)


print("Bayesian data shape:", bayesian_data.shape)

display(bayesian_data.head())

Bayesian data shape: (15000, 11)


,missing_data,invalid_date,future_request,approval_before_request,invalid_validity_range,negative_quantity,negative_amount,unusual_quantity,unusual_amount,duplicate_record,anomaly
0,no,no,no,no,no,no,no,no,no,no,no
1,no,no,no,no,no,no,no,no,no,no,no
2,no,no,no,no,no,no,no,no,no,no,no
3,no,no,no,no,no,no,no,no,no,no,no
4,no,no,no,no,no,no,no,no,no,no,no


In [29]:
for col in bayesian_data.columns:
    
    print("\n" + "=" * 50)
    print(col)

    print(
        bayesian_data[col]
        .value_counts()
    )


missing_data
missing_data
no    15000
Name: count, dtype: int64

invalid_date
invalid_date
no    15000
Name: count, dtype: int64

future_request
future_request
no    15000
Name: count, dtype: int64

approval_before_request
approval_before_request
no    15000
Name: count, dtype: int64

invalid_validity_range
invalid_validity_range
no    15000
Name: count, dtype: int64

negative_quantity
negative_quantity
no    15000
Name: count, dtype: int64

negative_amount
negative_amount
no    15000
Name: count, dtype: int64

unusual_quantity
unusual_quantity
no     11469
yes     3531
Name: count, dtype: int64

unusual_amount
unusual_amount
no     14235
yes      765
Name: count, dtype: int64

duplicate_record
duplicate_record
no    15000
Name: count, dtype: int64

anomaly
anomaly
no     14250
yes      750
Name: count, dtype: int64


In [30]:
edges = [

    ("missing_data", "anomaly"),
    ("invalid_date", "anomaly"),
    ("future_request", "anomaly"),

    ("approval_before_request", "anomaly"),
    ("invalid_validity_range", "anomaly"),

    ("negative_quantity", "anomaly"),
    ("negative_amount", "anomaly"),

    ("unusual_quantity", "anomaly"),
    ("unusual_amount", "anomaly"),

    ("duplicate_record", "anomaly")
]


model = BayesianNetwork(edges)

print("Bayesian Network created successfully.")

print("\nNodes:")
print(list(model.nodes()))

print("\nEdges:")
print(list(model.edges()))

Bayesian Network created successfully.

Nodes:
['missing_data', 'anomaly', 'invalid_date', 'future_request', 'approval_before_request', 'invalid_validity_range', 'negative_quantity', 'negative_amount', 'unusual_quantity', 'unusual_amount', 'duplicate_record']

Edges:
[('missing_data', 'anomaly'), ('invalid_date', 'anomaly'), ('future_request', 'anomaly'), ('approval_before_request', 'anomaly'), ('invalid_validity_range', 'anomaly'), ('negative_quantity', 'anomaly'), ('negative_amount', 'anomaly'), ('unusual_quantity', 'anomaly'), ('unusual_amount', 'anomaly'), ('duplicate_record', 'anomaly')]


In [31]:
model.fit(
    bayesian_data,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Bayesian Network trained successfully.")

Bayesian Network trained successfully.


In [32]:
inference = VariableElimination(model)

print("Bayesian inference engine created.")

Bayesian inference engine created.


In [36]:
baseline_probabilities = {}

for cause in root_cause_cols:

    result = inference.query(
        variables=[cause],
        show_progress=False
    )

    # Get the state names safely
    states = list(result.state_names[cause])

    # Find probability of "yes"
    if "yes" in states:

        yes_index = states.index("yes")

        baseline_probabilities[cause] = float(
            result.values[yes_index]
        )

    else:
        # If "yes" does not exist in this variable
        baseline_probabilities[cause] = 0.0


print("BASELINE PROBABILITIES")

for cause, probability in baseline_probabilities.items():

    print(
        f"{cause}: {probability:.4f}"
    )

BASELINE PROBABILITIES
missing_data: 0.0000
invalid_date: 0.0000
future_request: 0.0000
approval_before_request: 0.0000
invalid_validity_range: 0.0000
negative_quantity: 0.0000
negative_amount: 0.0000
unusual_quantity: 0.2356
unusual_amount: 0.0513
duplicate_record: 0.0000


In [37]:
def get_bayesian_root_causes(row, top_n=3):
    
    probable_causes = []


    for cause in root_cause_cols:

        # Cause must actually be present
        # in the current record

        if row[cause] != "yes":
            continue


        # Calculate:
        # P(cause=yes | anomaly=yes)

        result = inference.query(
            variables=[cause],
            evidence={
                "anomaly": "yes"
            },
            show_progress=False
        )


        states = result.state_names[cause]

        yes_index = states.index("yes")


        probability_given_anomaly = float(
            result.values[yes_index]
        )


        baseline_probability = (
            baseline_probabilities[cause]
        )


        # Bayesian lift

        if baseline_probability > 0:

            lift = (
                probability_given_anomaly
                /
                baseline_probability
            )

        else:

            lift = 0


        probable_causes.append({

            "cause": cause,

            "probability_given_anomaly":
                round(
                    probability_given_anomaly,
                    4
                ),

            "baseline_probability":
                round(
                    baseline_probability,
                    4
                ),

            "bayesian_lift":
                round(
                    lift,
                    2
                )
        })


    # Strongest probable causes first

    probable_causes = sorted(
        probable_causes,
        key=lambda x: (
            x["bayesian_lift"],
            x["probability_given_anomaly"]
        ),
        reverse=True
    )


    return probable_causes[:top_n]

In [38]:
bayesian_root_causes = []


for index, row in conditions.iterrows():

    if row["ml_anomaly_flag"] == 1:

        causes = get_bayesian_root_causes(
            row,
            top_n=3
        )

    else:

        causes = []


    bayesian_root_causes.append(causes)


conditions["bayesian_root_causes"] = (
    bayesian_root_causes
)

print("Bayesian root-cause analysis completed.")

Bayesian root-cause analysis completed.


In [39]:
cause_descriptions = {

    "missing_data":
        "One or more critical fields are missing",

    "invalid_date":
        "One or more date values have an invalid format",

    "future_request":
        "The request date is in the future",

    "approval_before_request":
        "The approval date occurs before the request date",

    "invalid_validity_range":
        "The validity end date occurs before the validity start date",

    "negative_quantity":
        "The requested quantity is negative",

    "negative_amount":
        "The charged amount is negative",

    "unusual_quantity":
        "The requested quantity is statistically unusual",

    "unusual_amount":
        "The charged amount is statistically unusual",

    "duplicate_record":
        "The record is duplicated"
}

In [40]:
def create_rag_text(row):
    
    if row["final_anomaly_flag"] == 0:

        return (
            "No anomaly was detected. "
            "The record passed configured data quality checks "
            "and was not identified as anomalous by Isolation Forest."
        )


    explanations = []


    # RULE-BASED ROOT CAUSES

    if row["rule_anomaly_flag"] == 1:

        rule_causes = []

        for cause in root_cause_cols:

            if row[cause] == "yes":

                rule_causes.append(
                    cause_descriptions[cause]
                )


        if rule_causes:

            explanations.append(
                "Rule-based data quality issues detected: "
                + "; ".join(rule_causes)
                + "."
            )


    # ML + BAYESIAN ROOT CAUSES

    if row["ml_anomaly_flag"] == 1:

        causes = row["bayesian_root_causes"]


        if len(causes) > 0:

            ml_causes = []


            for item in causes:

                cause = item["cause"]

                description = cause_descriptions.get(
                    cause,
                    cause.replace("_", " ")
                )


                ml_causes.append(
                    f"{description} "
                    f"(P(cause|anomaly)="
                    f"{item['probability_given_anomaly']}, "
                    f"Bayesian lift="
                    f"{item['bayesian_lift']})"
                )


            explanations.append(
                "Isolation Forest detected an unusual multivariate "
                "pattern. Bayesian analysis identified these probable "
                "contributing factors: "
                + "; ".join(ml_causes)
                + "."
            )

        else:

            explanations.append(
                "Isolation Forest detected an unusual multivariate "
                "pattern, but none of the configured root-cause "
                "conditions were present. This may indicate an "
                "unknown combination of feature values."
            )


    return " ".join(explanations)

In [41]:
conditions["rag_anomaly_context"] = (
    conditions.apply(
        create_rag_text,
        axis=1
    )
)

print("RAG-ready explanations generated.")

RAG-ready explanations generated.


In [42]:
display(
    conditions[
        conditions["final_anomaly_flag"] == 1
    ][
        [
            "rule_anomaly_flag",
            "ml_anomaly_flag",
            "ml_anomaly_score",
            "bayesian_root_causes",
            "rag_anomaly_context"
        ]
    ].head(10)
)

,rule_anomaly_flag,ml_anomaly_flag,ml_anomaly_score,bayesian_root_causes,rag_anomaly_context
6,1,0,-0.072222,[],Rule-based data quality issues detected: The c...
7,1,0,-0.095953,[],Rule-based data quality issues detected: The r...
9,1,0,-0.051050,[],Rule-based data quality issues detected: The r...
15,1,0,-0.028095,[],Rule-based data quality issues detected: The c...
17,1,0,-0.076064,[],Rule-based data quality issues detected: The r...
27,1,0,-0.085407,[],Rule-based data quality issues detected: The r...
30,1,0,-0.093514,[],Rule-based data quality issues detected: The r...
31,1,0,-0.090351,[],Rule-based data quality issues detected: The r...
34,1,0,-0.013663,[],Rule-based data quality issues detected: The r...
39,1,0,-0.078057,[],Rule-based data quality issues detected: The r...


In [43]:
def create_rag_json(index, row):
    
    rule_causes = []


    for cause in root_cause_cols:

        if row[cause] == "yes":

            rule_causes.append({

                "cause": cause,

                "description":
                    cause_descriptions[cause]
            })


    return {

        "authorization_id": str(
            df_original.loc[
                index,
                "authorization_id"
            ]
        ),

        "anomaly_detected": bool(
            row["final_anomaly_flag"] == 1
        ),

        "detection_sources": {

            "rule_based": bool(
                row["rule_anomaly_flag"] == 1
            ),

            "isolation_forest": bool(
                row["ml_anomaly_flag"] == 1
            )
        },

        "ml_anomaly_score": round(
            float(row["ml_anomaly_score"]),
            6
        ),

        "rule_based_root_causes":
            rule_causes,

        "bayesian_probable_root_causes":
            row["bayesian_root_causes"],

        "context_for_rag":
            row["rag_anomaly_context"]
    }

In [44]:
rag_json_list = []


for index, row in conditions.iterrows():

    result = create_rag_json(
        index,
        row
    )

    rag_json_list.append(result)


conditions["rag_json"] = [
    json.dumps(
        item,
        default=str
    )
    for item in rag_json_list
]

print("JSON column generated.")

JSON column generated.


In [45]:
final_df = pd.concat(
    [
        df_original,
        features.add_prefix("feature_"),
        conditions
    ],
    axis=1
)

print("Final shape:", final_df.shape)

display(final_df.head())

Final shape: (15000, 59)


,authorization_id,patient_id,provider_id,request_date,authorization_type,service_code,service_description,requested_quantity,charged_amount,approval_status,...,unusual_amount,duplicate_record,rule_anomaly_count,rule_anomaly_flag,ml_anomaly_flag,ml_anomaly_score,final_anomaly_flag,bayesian_root_causes,rag_anomaly_context,rag_json
0,AUTH00001-2026A,PAT58368,PRV4993,2025-03-31,medication,NDC43598-0545-01,Albuterol inhaler,2,187.35,approved,...,no,no,0,0,0,-0.096876,0,[],No anomaly was detected. The record passed con...,"{""authorization_id"": ""AUTH00001-2026A"", ""anoma..."
1,AUTH00002-2026A,PAT25525,PRV8245,2025-12-05,DME,HCPCS E0165,Commode chair,1,338.48,approved,...,no,no,0,0,0,-0.082698,0,[],No anomaly was detected. The record passed con...,"{""authorization_id"": ""AUTH00002-2026A"", ""anoma..."
2,AUTH00003-2026A,PAT35112,PRV3165,2026-02-04,procedure,CPT99214,"Office visit, established patient",1,362.78,approved,...,no,no,0,0,0,-0.107327,0,[],No anomaly was detected. The record passed con...,"{""authorization_id"": ""AUTH00003-2026A"", ""anoma..."
3,AUTH00004-2026A,PAT42177,PRV4073,2026-04-14,DME,HCPCS E0607,CPAP device,1,2007.68,denied,...,no,no,0,0,0,-0.087366,0,[],No anomaly was detected. The record passed con...,"{""authorization_id"": ""AUTH00004-2026A"", ""anoma..."
4,AUTH00005-2026A,PAT71433,PRV3634,2025-01-27,procedure,CPT29881,Knee arthroscopy with meniscectomy,1,5304.87,approved,...,no,no,0,0,0,-0.086950,0,[],No anomaly was detected. The record passed con...,"{""authorization_id"": ""AUTH00005-2026A"", ""anoma..."


In [46]:
OUTPUT_FILE = "authorization_anomaly_results.csv"

final_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("Complete results saved successfully!")
print("File:", OUTPUT_FILE)
print("Total records:", len(final_df))

Complete results saved successfully!
File: authorization_anomaly_results.csv
Total records: 15000


In [47]:
rag_output_df = final_df[
    final_df["final_anomaly_flag"] == 1
].copy()


RAG_OUTPUT_FILE = (
    "authorization_anomalies_for_rag.csv"
)


rag_output_df.to_csv(
    RAG_OUTPUT_FILE,
    index=False
)


print("RAG CSV saved successfully!")
print("File:", RAG_OUTPUT_FILE)
print("Anomaly records:", len(rag_output_df))

RAG CSV saved successfully!
File: authorization_anomalies_for_rag.csv
Anomaly records: 4497


In [48]:
json_output = []


for index, row in conditions.iterrows():

    if row["final_anomaly_flag"] == 1:

        json_output.append(
            create_rag_json(
                index,
                row
            )
        )


JSON_OUTPUT_FILE = (
    "authorization_anomalies_for_rag.json"
)


with open(
    JSON_OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        json_output,
        f,
        indent=4,
        default=str
    )


print("JSON file saved successfully!")
print("File:", JSON_OUTPUT_FILE)
print("Anomaly records:", len(json_output))

JSON file saved successfully!
File: authorization_anomalies_for_rag.json
Anomaly records: 4497


In [50]:
import os
import pickle

# Create the folder if it doesn't exist
os.makedirs("pkl file", exist_ok=True)

PIPELINE_PATH = "pkl file/authorization_anomaly_pipeline.pkl"

with open(PIPELINE_PATH, "wb") as f:
    pickle.dump(authorization_pipeline, f)

print("Pipeline saved successfully!")
print("Path:", PIPELINE_PATH)

Pipeline saved successfully!
Path: pkl file/authorization_anomaly_pipeline.pkl


In [51]:
import pickle
import os

# Create models folder
os.makedirs("models", exist_ok=True)


# ==========================================
# COMPLETE AUTHORIZATION ANOMALY PIPELINE
# ==========================================

authorization_pipeline = {

    # --------------------------------------
    # TRAINED ML MODEL
    # --------------------------------------
    "isolation_forest_model": iso_model,


    # --------------------------------------
    # PREPROCESSING OBJECTS
    # --------------------------------------
    "scaler": scaler,
    "encoder": encoder,


    # --------------------------------------
    # BAYESIAN ROOT CAUSE MODEL
    # --------------------------------------
    "bayesian_network": model,
    "baseline_probabilities": baseline_probabilities,


    # --------------------------------------
    # FEATURE INFORMATION
    # IMPORTANT: Maintains exact training order
    # --------------------------------------
    "ml_feature_columns": ml_df.columns.tolist(),

    "categorical_features": categorical_features,

    "categorical_cols": categorical_cols,

    "numeric_cols": numeric_cols,


    # --------------------------------------
    # DATA / RULE CONFIGURATION
    # --------------------------------------
    "input_columns": df.columns.tolist(),

    "date_columns": date_cols,

    "critical_cols": critical_cols,

    "root_cause_columns": root_cause_cols,


    # --------------------------------------
    # IQR THRESHOLDS
    # Needed for future rule detection
    # --------------------------------------
    "quantity_lower": quantity_lower,
    "quantity_upper": quantity_upper,

    "amount_lower": amount_lower,
    "amount_upper": amount_upper,


    # --------------------------------------
    # ROOT CAUSE DESCRIPTIONS
    # --------------------------------------
    "cause_descriptions": cause_descriptions
}


# ==========================================
# SAVE AS ONE PKL FILE
# ==========================================

PIPELINE_PATH = "pkl file/authorization_anomaly_pipeline.pkl"

with open(PIPELINE_PATH, "wb") as f:
    pickle.dump(authorization_pipeline, f)


print("=" * 60)
print("COMPLETE PIPELINE SAVED SUCCESSFULLY")
print("=" * 60)

print("\nPipeline path:")
print(PIPELINE_PATH)

print("\nPipeline components:")

for key in authorization_pipeline.keys():
    print("-", key)

COMPLETE PIPELINE SAVED SUCCESSFULLY

Pipeline path:
pkl file/authorization_anomaly_pipeline.pkl

Pipeline components:
- isolation_forest_model
- scaler
- encoder
- bayesian_network
- baseline_probabilities
- ml_feature_columns
- categorical_features
- categorical_cols
- numeric_cols
- input_columns
- date_columns
- critical_cols
- root_cause_columns
- quantity_lower
- quantity_upper
- amount_lower
- amount_upper
- cause_descriptions


In [52]:
import pickle

with open(
    "pkl file/authorization_anomaly_pipeline.pkl",
    "rb"
) as f:
    
    loaded_pipeline = pickle.load(f)


print("Pipeline loaded successfully!\n")

print("Available components:")

for key in loaded_pipeline.keys():
    print("-", key)


print("\nIsolation Forest type:")
print(type(loaded_pipeline["isolation_forest_model"]))

print("\nBayesian Network type:")
print(type(loaded_pipeline["bayesian_network"]))

Pipeline loaded successfully!

Available components:
- isolation_forest_model
- scaler
- encoder
- bayesian_network
- baseline_probabilities
- ml_feature_columns
- categorical_features
- categorical_cols
- numeric_cols
- input_columns
- date_columns
- critical_cols
- root_cause_columns
- quantity_lower
- quantity_upper
- amount_lower
- amount_upper
- cause_descriptions

Isolation Forest type:
<class 'sklearn.ensemble._iforest.IsolationForest'>

Bayesian Network type:
<class 'pgmpy.models.BayesianNetwork.BayesianNetwork'>
